**Assignment-8**

Implement a pre-trained BERT model for sentiment analysis or text classification on a sample dataset.

Dataset used: Amazon Polarity dataset (Hugging Face Datasets Hub)


In [29]:
# Install required dependencies for NLP, deep learning, and evaluation
!pip -q install -U transformers datasets accelerate scikit-learn

In [30]:
# Import libraries, set random seeds for reproducibility, and configure computing device (GPU/CPU)
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [31]:
# Load the Amazon Polarity dataset and sample subset splits for training and testing
dataset = load_dataset('mteb/amazon_polarity')

# Sample sizes: increase these for better accuracy on a GPU
TRAIN_SIZE = 2000
TEST_SIZE = 500

train_ds = dataset['train'].shuffle(seed=SEED).select(range(TRAIN_SIZE))
test_ds = dataset['test'].shuffle(seed=SEED).select(range(TEST_SIZE))

print('Training examples:', len(train_ds))
print('Testing examples : ', len(test_ds))
print('\nExample review:\n', train_ds[0]['text'][:500])
print('Label:', train_ds[0]['label'])

Training examples: 2000
Testing examples :  500

Example review:
 Amazon should apologize for Kindle delays over Christmas!!!

With all the hype about ordering Kindle for the holidays I ordered early - in November, figuring I'd give it a shot despite the high price and semi-negative feedback. So, imagine my dismay with a week to go before Christmas - and not only has the thing never shipped - I now get a little cheerful snowflake-clad announcement that it won't even arrive until AFTER Christmas! A MONTH AFTER I ORDERED IT! Maybe Amazon shouldn't be hyping thei
Label: 0


In [32]:
# Initialize the BERT tokenizer and process the text datasets into tokenized inputs
MODEL_NAME = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256
    )

tokenized_train = train_ds.map(tokenize_function, batched=True)
tokenized_test = test_ds.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenizer('I really enjoyed this movie!'))

{'input_ids': [101, 1045, 2428, 5632, 2023, 3185, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [33]:
# Load pre-trained BERT with a sequence classification head and map class labels
id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print('Number of trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of trainable parameters: 109483778


In [34]:
# Define a helper function to calculate accuracy, precision, recall, and F1 score during model evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [35]:
# Configure hyperparameters and initialize the Trainer to fine-tune BERT on our training dataset
training_args = TrainingArguments(
    output_dir='./bert_sentiment_results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.257918,0.237059,0.922000,0.922764,0.919028,0.920892


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=250, training_loss=0.33856479263305667, metrics={'train_runtime': 91.478, 'train_samples_per_second': 21.863, 'train_steps_per_second': 2.733, 'total_flos': 193267403492640.0, 'train_loss': 0.33856479263305667, 'epoch': 1.0})

In [36]:
# Run the trained model on the validation dataset and print overall performance metrics
metrics = trainer.evaluate()

print('Evaluation Results')
print('-' * 40)
for key, value in metrics.items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.257918,0.237059,1,0.922000,0.922764,0.919028,0.920892


Evaluation Results
----------------------------------------
eval_loss: 0.2371
eval_accuracy: 0.9220
eval_precision: 0.9228
eval_recall: 0.9190
eval_f1: 0.9209


In [37]:
# Generate detailed predictions and print a structured classification report (precision, recall, F1-score per class)
pred_output = trainer.predict(tokenized_test)
predictions = np.argmax(pred_output.predictions, axis=-1)
true_labels = pred_output.label_ids

print(classification_report(
    true_labels,
    predictions,
    target_names=['NEGATIVE', 'POSITIVE'],
    digits=4
))

              precision    recall  f1-score   support

    NEGATIVE     0.9213    0.9249    0.9231       253
    POSITIVE     0.9228    0.9190    0.9209       247

    accuracy                         0.9220       500
   macro avg     0.9220    0.9220    0.9220       500
weighted avg     0.9220    0.9220    0.9220       500



In [38]:
# Define a custom inference function to predict sentiment labels and confidence scores on new sentences
def predict_sentiment(texts):
    model.eval()
    inputs = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

    results = []
    for text, pred, probs in zip(texts, predictions, probabilities):
        results.append({
            'text': text,
            'sentiment': id2label[pred.item()],
            'confidence': float(probs[pred].item())
        })
    return results

sample_texts = [
    'This movie was absolutely fantastic. I loved every minute of it!',
    'The story was boring and the acting was terrible.',
    'It was an enjoyable movie with excellent performances.'
]

results = predict_sentiment(sample_texts)
for result in results:
    print(f"\nText      : {result['text']}")
    print(f"Sentiment : {result['sentiment']}")
    print(f"Confidence: {result['confidence']:.2%}")


Text      : This movie was absolutely fantastic. I loved every minute of it!
Sentiment : POSITIVE
Confidence: 98.70%

Text      : The story was boring and the acting was terrible.
Sentiment : NEGATIVE
Confidence: 96.86%

Text      : It was an enjoyable movie with excellent performances.
Sentiment : POSITIVE
Confidence: 98.51%


In [39]:
# Export and save the fine-tuned model weights and its vocabulary/tokenizer configuration to disk
SAVE_PATH = './bert_sentiment_model'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print('Model saved to:', SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./bert_sentiment_model


### Key Concepts

- **BERT**: A pre-trained Transformer language model that understands contextual relationships between words.
- **Fine-tuning**: Adapting the pre-trained BERT model to a specific task using labelled examples.
- **Tokenizer**: Converts text into BERT-compatible token IDs.
- **Sequence classification head**: Converts BERT representations into class predictions.
- **Softmax**: Converts output logits into class probabilities.
- **Accuracy / Precision / Recall / F1**: Metrics used to evaluate classification performance.


### Workflow
1. Install required libraries
2. Load a sample of the IMDB dataset
3. Tokenize text using the BERT tokenizer
4. Fine-tune pre-trained BERT for sentiment classification
5. Evaluate using Accuracy, Precision, Recall and F1-score
6. Test the trained model on new sentences


### About Dataset
A massive collection of Amazon product reviews categorized into binary positive and negative sentiment.